# S3_NB3 — how much of the gap can a learned router capture?

**~5 GPU-hours · feature dump + a small gate per exit**

## The overclaim this fixes

Study 2 says the oracle excess "cannot be reached by any router". What was
actually shown is that **a second seed** cannot reach it. A learned router with
access to the input might do better, and nobody has measured it.

```
capture fraction = (router − confidence baseline) / (oracle_in − baseline)
```

**Pre-registered (H2):** a learned router captures **< 25 %** of the gap.

| outcome | reading |
|---|---|
| captures most | the field is right, the gap is real headroom, and here is a router |
| captures a little | the bound is mostly noise, now quantified |
| captures none | the strongest version of Study 2's claim |

All three are reportable and two are positive.

## The deployability constraint

A gate at exit *k* may use **only features available at exit k**. Anything else
is not a router, it is an oracle wearing a router's clothes — the exact mistake
`pred_depth` turned out to be in Study 2.

## The control that decides whether the number means anything

Train the gate on seed *i*, evaluate on seed *j*'s network. An in-seed capture
fraction alone is uninterpretable: a gate can fit one seed's noise perfectly.
**Both numbers are reported, always.**

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   7c88c31b2076   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  2cc4ba5e0935   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

In [ ]:
# === Who am I? =============================================================
#
# ACCOUNT   labels this Kaggle account in the run log. Two accounts calling
#           themselves the same thing makes the log useless.
# WORKER_ID splits the work. Every account runs THE SAME notebook; the only
#           thing that differs is this number. Account 1 -> 0, account 2 -> 1,
#           and so on. Each account then works out, by pure arithmetic, which
#           jobs belong to it -- no communication needed, no chance of two
#           accounts training the same model, no chance of a job being missed.
#
# DEFAULT IS 1: this account does everything in this notebook. That is the
# simplest thing that works. Change it only when you actually have several
# accounts running at once.
# Study 3 Q2. Feature dump + gate training.
ACCOUNT     = 'acct1'      # <<< CHANGE ME
NUM_WORKERS = 1          # <<< how many accounts you are running in parallel
WORKER_ID   = 0            # <<< CHANGE ME: 0, 1, 2, ... up to NUM_WORKERS-1

sess = msc.Session(account=ACCOUNT, phase='p4', dataset='cifar100',
                   worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                   shard_mode='cost',        # balance GPU-hours, not job counts
                   enable_hf=True,
                   session_limit_h=8.5,      # push + pause before Kaggle kills us
                   commits_per_hour_limit=20,# 6 accounts x 20 = 120 < HF's 128/hr
                   batch_interval_sec=1800)  # the 30-minute push policy

In [ ]:
# === Names ================================================================
# `msc` is how the CIFAR-100 notebooks import the library; `M` is how Study 2's
# notebooks refer to it. Same module, two names, declared here once so no cell
# below has to guess which convention it is in.
M = msc
MSC_ROOT = sess.work          # the results root: runs/, analysis/, budgets/

print(f'msc_lib {M.__version__}')
print(f'MSC_ROOT  {MSC_ROOT}')
print(f'data_dir  {sess.data_dir}')
assert (Path(MSC_ROOT) / 'runs').is_dir(), (
    f'no runs/ under {MSC_ROOT} -- fetch Study 1 data first (S2_NB0_Fetch)')

In [ ]:
# === Get CIFAR-100 =========================================================
# Looks in this order: attached Kaggle dataset (instant) -> earlier extraction
# -> Kaggle CLI download -> torchvision as a last resort.
# Everything lands in /kaggle/temp (~1 TB scratch), never in /kaggle/working
# (20 GB, and that is the space your results need).
DATA_ROOT = sess.prepare_data()
print('dataset:', DATA_ROOT)

---
## Dump per-exit features

One forward pass per run. ~50 MB per run at 256 dims — cheap, and it is the
only thing standing between us and a learned router.

In [ ]:

import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path

ARCHS = ['resnet20', 'resnet32x4', 'vgg8']
SEEDS = [1, 2]          # two seeds: the cross-seed control needs them
feat_dir = Path(MSC_ROOT) / 'features'
feat_dir.mkdir(parents=True, exist_ok=True)

def dump_features(run_id, cfg):
    out = feat_dir / f'{run_id}.npz'
    if out.exists():
        return out
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    L = M.run_layout(sess.work, run_id)
    blob = torch.load(L['checkpoints'] / 'ckpt_best.pt', map_location=device,
                      weights_only=False)
    backbone = M.place_model(M.build_model(cfg['arch'], cfg['num_classes']),
                             device, cfg, tag='feature dump')
    backbone.load_state_dict(blob['model'], strict=True)
    me = M.place_model(M.MultiExitModel(backbone, cfg['num_classes'], freeze=True),
                       device, cfg)
    hp = M.exit_heads_path(sess.work, run_id)
    me.heads.load_state_dict(torch.load(hp, map_location=device,
                                        weights_only=False)['heads'])
    me.eval()

    _, val_loader, _, _, _ = M.build_loaders(cfg)
    feats, labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch[0].to(device, non_blocking=True)
            fs = me.backbone.forward_features(x)
            pooled = []
            for f in fs:
                if f.dim() == 4:
                    pooled.append(nn.functional.adaptive_avg_pool2d(f, 1)
                                  .flatten(1).float().cpu().numpy())
                elif f.dim() == 3:
                    pooled.append((f[:, 0] if me.token_model
                                   else f.mean(1)).float().cpu().numpy())
                else:
                    pooled.append(f.flatten(1).float().cpu().numpy())
            feats.append(pooled)
            labels.append(batch[1].numpy())
    K = len(feats[0])
    stacked = {f'f{k}': np.concatenate([b[k] for b in feats], axis=0)
               for k in range(K)}
    np.savez_compressed(out, label=np.concatenate(labels), **stacked)
    del me, backbone
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

paths = {}
for a in ARCHS:
    for s in SEEDS:
        cfg = M.base_config(a, 'cifar100', seed=s, phase='p1', method='base')
        rid = cfg['run_id']
        if not (Path(MSC_ROOT) / 'runs' / rid / 'per_sample' / 'test.parquet').exists():
            print(f'  missing {rid}, skipped')
            continue
        paths[rid] = dump_features(rid, cfg)
        mb = paths[rid].stat().st_size / 2**20
        print(f'  {rid:38s} {mb:6.1f} MB')
print(f'\n{len(paths)} feature dump(s)')

---
## Train a gate per exit, then measure capture

The gate is deliberately small. A large one would fit the seed's noise and
inflate the in-seed number — which the cross-seed control would then expose,
but it is cheaper not to invite the problem.

In [ ]:

from sklearn.linear_model import LogisticRegression   # small on purpose

def gate_scores(train_rid, eval_rid):
    '''Train per-exit gates on train_rid, score eval_rid's samples.'''
    tr = np.load(paths[train_rid]); ev = np.load(paths[eval_rid])
    dtr = pd.read_parquet(Path(MSC_ROOT) / 'runs' / train_rid
                          / 'per_sample' / 'test.parquet').sort_values('sample_idx')
    ks = sorted(int(c.split('_d')[1]) for c in dtr.columns
                if c.startswith('pred_d') and c.split('_d')[1].isdigit())
    lab_tr = dtr['label'].to_numpy()
    out = []
    for i, k in enumerate(ks[:-1]):        # no gate needed at the final exit
        y = (dtr[f'pred_d{k}'].to_numpy() == lab_tr).astype(int)
        Xtr, Xev = tr[f'f{i}'], ev[f'f{i}']
        if y.min() == y.max():
            out.append(np.full(len(Xev), float(y.mean())))
            continue
        clf = LogisticRegression(max_iter=300, C=0.1)
        clf.fit(Xtr, y)
        out.append(clf.predict_proba(Xev)[:, 1])
    return np.stack(out, axis=1), ks

def correctness(rid):
    d = pd.read_parquet(Path(MSC_ROOT) / 'runs' / rid / 'per_sample'
                        / 'test.parquet').sort_values('sample_idx')
    ks = sorted(int(c.split('_d')[1]) for c in d.columns
                if c.startswith('pred_d') and c.split('_d')[1].isdigit())
    lab = d['label'].to_numpy()
    corr = np.stack([(d[f'pred_d{k}'].to_numpy() == lab) for k in ks], axis=1).astype(float)
    conf = np.stack([d[f'top1p_d{k}'].to_numpy() for k in ks], axis=1)
    return corr, conf, ks

---
## Evaluate at matched budget

Routing helpers are **imported from Study 2's notebook logic**, re-implemented
here only because the notebook is standalone — but the canaries in
`tools/s2_routing_canaries.py` cover the same functions and must pass first.

In [ ]:

def _cost(k, rho):  return float(np.mean(np.asarray(rho)[k]))

def route_confidence(conf, correct, rho, target):
    n, K = correct.shape
    lo, hi = 0.0, 1.0
    for _ in range(60):
        th = (lo + hi) / 2
        fires = conf >= th; fires[:, -1] = True
        k = fires.argmax(axis=1); c = _cost(k, rho)
        if c < target: lo = th
        else: hi = th
    return float(correct[np.arange(n), k].mean()), c

def route_gate(pgate, correct, rho, target):
    '''Exit at the first exit whose gate probability clears a threshold.'''
    n, K = correct.shape
    lo, hi = 0.0, 1.0
    for _ in range(60):
        th = (lo + hi) / 2
        fires = np.concatenate([pgate >= th, np.ones((n, 1), bool)], axis=1)
        k = fires.argmax(axis=1); c = _cost(k, rho)
        if c < target: lo = th
        else: hi = th
    return float(correct[np.arange(n), k].mean()), c

def route_oracle(cc, ce, rho, target):
    rho = np.asarray(rho, float)
    lo, hi = 0.0, 100.0
    for _ in range(80):
        lam = (lo + hi) / 2
        k = (cc - lam * rho[None, :]).argmax(axis=1)
        if float(rho[k].mean()) > target: lo = lam
        else: hi = lam
    k = (cc - hi * rho[None, :]).argmax(axis=1)
    return float(ce[np.arange(len(ce)), k].mean()), float(rho[k].mean())

TARGET = 0.80
rows = []
for a in ARCHS:
    rids = [r for r in paths if M.parse_run_id(r)['arch'] == a]
    if len(rids) < 2:
        continue
    rho = M.load_or_build_budgets(a, sess.work, 'cifar100')['axes']['depth']['rho']
    i, j = sorted(rids)[0], sorted(rids)[1]
    for train_on, eval_on, kind in [(i, i, 'in-seed'), (i, j, 'cross-seed')]:
        corr, conf, ks = correctness(eval_on)
        base, _ = route_confidence(conf, corr, rho, TARGET)
        orac, _ = route_oracle(corr, corr, rho, TARGET)
        pg, _ = gate_scores(train_on, eval_on)
        gt, _ = route_gate(pg, corr, rho, TARGET)
        gap = orac - base
        rows.append({'arch': a, 'kind': kind, 'baseline': base * 100,
                     'router': gt * 100, 'oracle': orac * 100,
                     'gap': gap * 100, 'router_gain': (gt - base) * 100,
                     'capture': (gt - base) / gap if gap > 1e-9 else np.nan})

cap = pd.DataFrame(rows)
M.save_analysis(sess.data_dir, 's3_router_capture', cap)
print(cap.round(3).to_string(index=False))
print()
for kind in ['in-seed', 'cross-seed']:
    sub = cap[cap['kind'] == kind]
    if len(sub):
        print(f'  {kind:11s} median capture = {sub["capture"].median()*100:6.2f} %')
print()
cs = cap[cap['kind'] == 'cross-seed']['capture'].median()
print(f'H2 (< 25 % captured): '
      f'{"SUPPORTED" if cs < 0.25 else "FALSIFIED"}  (cross-seed {cs*100:.1f} %)')
print()
print('The CROSS-SEED number is the one that means anything. If in-seed capture')
print('is high and cross-seed is not, the gate memorised one seed s noise --')
print('which is Study 2 s finding restated, not a contradiction of it.')

---
## Canaries — the gate must be shown to work and to fail

In [ ]:

rng = np.random.default_rng(0)
n, K = 4000, 5
rho = [0.2, 0.4, 0.6, 0.8, 1.0]
easy = rng.random(n) < 0.5
cc = np.zeros((n, K)); cc[easy, :] = 1.0; cc[~easy, K-1] = 1.0

# a gate handed the truth must capture ~everything
perfect = np.repeat(easy[:, None].astype(float), K-1, axis=1)
b, _ = route_confidence(rng.random((n, K)), cc, rho, 0.7)
o, _ = route_oracle(cc, cc, rho, 0.7)
g, _ = route_gate(perfect, cc, rho, 0.7)
capture = (g - b) / (o - b) if o > b else float('nan')
print(f'{"PASS" if capture > 0.8 else "FAIL"}  oracle-gate captures '
      f'{capture*100:.0f}% (must be ~100)')

# a gate handed noise must capture ~nothing
g2, _ = route_gate(rng.random((n, K-1)), cc, rho, 0.7)
cap2 = (g2 - b) / (o - b) if o > b else float('nan')
print(f'{"PASS" if cap2 < 0.25 else "FAIL"}  noise-gate captures '
      f'{cap2*100:.0f}% (must be ~0)')

---
## Next

Record in `study3/03_LOG.md`. Then `S3_NB4_Pruning` (Q3), which is independent
of everything above.